# Notebook 04 — Feature, ZigZag Timing, and Leakage Audit

## Purpose

This notebook freezes the model-input schema and converts the ZigZag design
intent into an explicitly confirmation-gated, prefix-invariant primary long-side
generator.

## Non-negotiable controls

- Feature approval uses **train only**.
- Unseen-test outcomes are not read for feature or threshold selection.
- Label, event-end, censoring, barrier, and legacy future-derived fields are never model inputs.
- Legacy `zigzag_up_new_2` and `zigzag_down_new_2` are audited but are **not**
  used directly as model features or as the primary event filter.
- The ZigZag state is reconstructed from adjusted price history and becomes
  available only at the pivot confirmation observation.
- The **15%** candidate threshold is the pre-registered primary rule.
- **10%** and **20%** are train-only sensitivity diagnostics.
- The confirmed ZigZag rule supplies `side = long`; within those candidate
  events, the frozen Triple-Barrier `label` is the take/skip meta-label.


In [1]:
from __future__ import annotations

from collections import defaultdict
from pathlib import Path
import json
import sys
import time

import numpy as np
import pandas as pd
import yaml

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("numpy:", np.__version__)


Python: 3.12.2
pandas: 2.2.3
numpy: 1.26.4


## 1. Locate the repository and import project modules

In [2]:
def locate_repository_root(start: Path) -> Path:
    current = start.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "configs").exists() and (candidate / "notebooks").exists():
            return candidate
    raise FileNotFoundError(
        "Repository root was not found. Run this notebook from inside the project."
    )


REPOSITORY_ROOT = locate_repository_root(Path.cwd())

if str(REPOSITORY_ROOT) not in sys.path:
    sys.path.insert(0, str(REPOSITORY_ROOT))

from src.features.feature_schema import (
    audit_prohibited_feature_names,
    build_feature_role_table,
    finalize_feature_approval,
)
from src.features.leakage_checks import (
    ConfirmedZigZagConfig,
    build_candidate_long_mask,
    build_confirmation_gated_zigzag_state,
    legacy_zigzag_source_logic_audit,
    prefix_invariance_audit,
)
from src.utils.paths import (
    data_paths,
    repository_result_paths,
    resolve_data_root,
)
from src.utils.reproducibility import (
    git_commit_sha,
    software_manifest,
    stable_object_hash,
)

print("Repository root:", REPOSITORY_ROOT)


Repository root: E:\Iran_stock_trade\financial-ml-decision-support


## 2. Load frozen Stage 03 policy and Stage 04 feature policy

In [3]:
def load_yaml(path: Path) -> dict:
    with path.open("r", encoding="utf-8") as file_obj:
        value = yaml.safe_load(file_obj)
    if not isinstance(value, dict):
        raise ValueError(f"Configuration must be a mapping: {path}")
    return value


paths_config = load_yaml(REPOSITORY_ROOT / "configs" / "paths.yaml")
columns_config = load_yaml(REPOSITORY_ROOT / "configs" / "columns.yaml")
labeling_config = load_yaml(REPOSITORY_ROOT / "configs" / "labeling.yaml")
validation_config = load_yaml(REPOSITORY_ROOT / "configs" / "validation.yaml")

DATA_ROOT = resolve_data_root(paths_config, REPOSITORY_ROOT)
DATA_PATHS = data_paths(DATA_ROOT, paths_config)
RESULT_PATHS = repository_result_paths(REPOSITORY_ROOT, paths_config)

LABELED_TRAIN_PATH = DATA_PATHS["labeled_data"] / "train"
CANDIDATE_TRAIN_PATH = DATA_PATHS["candidate_data"] / "train"
CANDIDATE_TRAIN_PATH.mkdir(parents=True, exist_ok=True)

DATE_COLUMN = "dEven"
OUTPUT_ENCODING = "utf-8-sig"

feature_policy = columns_config["feature_policy"]
zigzag_policy = columns_config["zigzag_audit"]
candidate_policy = columns_config["candidate_event_filter"]

assert labeling_config["status"] == "stage_03_frozen"
assert labeling_config["frozen_for_experiment"] is True
assert labeling_config["selected_scenario"] == "symmetric_15_15"
assert labeling_config["zigzag_policy"]["used_to_construct_label"] is False
assert labeling_config["zigzag_policy"]["formal_row_level_audit_stage"] == "Notebook 04"

PRIMARY_THRESHOLD = float(candidate_policy["primary_threshold_fraction"])
SENSITIVITY_THRESHOLDS = [
    float(value)
    for value in candidate_policy["sensitivity_threshold_fractions"]
]
ALL_THRESHOLDS = sorted(set([PRIMARY_THRESHOLD, *SENSITIVITY_THRESHOLDS]))

assert candidate_policy["selection_scope"] == "train_only"
assert candidate_policy["automatic_threshold_selection"] is False
assert np.isclose(
    PRIMARY_THRESHOLD,
    float(labeling_config["zigzag_policy"]["intended_filter_threshold_fraction"]),
)
assert np.isclose(PRIMARY_THRESHOLD, 0.15)

zigzag_config = ConfirmedZigZagConfig(
    depth=int(zigzag_policy["source_depth"]),
    deviation_percent=float(zigzag_policy["source_deviation_percent"]),
)

print("Labeled train input:", LABELED_TRAIN_PATH)
print("Candidate train output:", CANDIDATE_TRAIN_PATH)
print("Primary filter threshold:", PRIMARY_THRESHOLD)
print("Sensitivity thresholds:", SENSITIVITY_THRESHOLDS)
print("Unseen-test used for Stage 04 approval: False")


Labeled train input: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\labeled\train
Candidate train output: E:\Iran_stock_trade\financial-ml-decision-support\data_ready\candidates\train
Primary filter threshold: 0.15
Sensitivity thresholds: [0.1, 0.2]
Unseen-test used for Stage 04 approval: False


## 3. Validate the frozen universe and Stage 03 labeled train files

In [4]:
frozen_universe_path = RESULT_PATHS["manifests"] / "02_frozen_universe.csv"
if not frozen_universe_path.exists():
    raise FileNotFoundError(
        "Frozen universe manifest is missing. Run and freeze Notebook 02 first."
    )

frozen_universe_df = pd.read_csv(frozen_universe_path, low_memory=False)
frozen_symbols = sorted(frozen_universe_df["symbol"].astype(str).tolist())
frozen_symbol_set = set(frozen_symbols)

train_files = sorted(LABELED_TRAIN_PATH.glob("*_train_labeled.csv"))

def symbol_from_path(path: Path) -> str:
    suffix = "_train_labeled.csv"
    if not path.name.endswith(suffix):
        raise ValueError(f"Unexpected file name: {path.name}")
    return path.name[:-len(suffix)]

train_file_map = {
    symbol_from_path(path): path
    for path in train_files
}

assert set(train_file_map) == frozen_symbol_set
assert len(train_files) == len(frozen_symbols)

print("Frozen universe size:", len(frozen_symbols))
print("Labeled train files:", len(train_files))


Frozen universe size: 499
Labeled train files: 499


## 4. Structural feature-role and prohibited-name audit

In [5]:
candidate_features = list(columns_config["candidate_features"])

role_table_df = build_feature_role_table(columns_config)
prohibited_name_audit_df = audit_prohibited_feature_names(
    candidate_features,
    list(feature_policy["prohibited_feature_name_patterns"]),
)

assert role_table_df["role_membership_count"].eq(1).all()
assert not prohibited_name_audit_df["prohibited_name_hit"].any()

role_table_path = RESULT_PATHS["audits"] / "04_feature_role_audit.csv"
prohibited_path = RESULT_PATHS["audits"] / "04_prohibited_name_audit.csv"

role_table_df.to_csv(role_table_path, index=False, encoding=OUTPUT_ENCODING)
prohibited_name_audit_df.to_csv(
    prohibited_path,
    index=False,
    encoding=OUTPUT_ENCODING,
)

display(role_table_df)


,feature,structural_role,structurally_approved,structural_reason,role_membership_count
0,adj_open,context_only,False,"Adjusted OHLC level retained for labeling, aud...",1
1,adj_high,context_only,False,"Adjusted OHLC level retained for labeling, aud...",1
2,adj_low,context_only,False,"Adjusted OHLC level retained for labeling, aud...",1
3,adj_last_price,context_only,False,"Adjusted OHLC level retained for labeling, aud...",1
4,zigzag_up_new_2,legacy_zigzag_rejected,False,Legacy new_2 ZigZag columns are replaced by a ...,1
5,zigzag_down_new_2,legacy_zigzag_rejected,False,Legacy new_2 ZigZag columns are replaced by a ...,1
6,EMA_3,model_candidate,True,Eligible for train-only data-quality approval.,1
7,EMA_5,model_candidate,True,Eligible for train-only data-quality approval.,1
8,EMA_8,model_candidate,True,Eligible for train-only data-quality approval.,1
9,EMA_10,model_candidate,True,Eligible for train-only data-quality approval.,1


## 5. Train-only feature quality audit

In [6]:
quality_accumulator = {
    feature: {
        "total_rows": 0,
        "missing_values": 0,
        "finite_values": 0,
        "nonfinite_values": 0,
        "global_min": np.inf,
        "global_max": -np.inf,
        "symbols_present": 0,
    }
    for feature in candidate_features
}

schema_error_rows = []
started = time.perf_counter()

required_columns = {
    DATE_COLUMN,
    "eligible_for_modeling",
    "label",
    "barrier_touched",
    "holding_period_observations",
    "event_return",
    "adj_high",
    "adj_low",
    "adj_last_price",
    *candidate_features,
}

for file_number, (symbol, path) in enumerate(
    sorted(train_file_map.items()),
    start=1,
):
    try:
        frame = pd.read_csv(path, low_memory=False)

        missing_columns = sorted(required_columns - set(frame.columns))
        if missing_columns:
            raise KeyError(
                f"{path.name}: missing required columns {missing_columns}"
            )

        eligible = frame["eligible_for_modeling"].fillna(False).astype(bool)
        eligible_frame = frame.loc[eligible]

        for feature in candidate_features:
            values = pd.to_numeric(
                eligible_frame[feature],
                errors="coerce",
            )
            array = values.to_numpy(dtype=float)
            finite = np.isfinite(array)

            target = quality_accumulator[feature]
            target["total_rows"] += int(len(array))
            target["missing_values"] += int(values.isna().sum())
            target["finite_values"] += int(finite.sum())
            target["nonfinite_values"] += int(
                np.isinf(array).sum()
            )
            target["symbols_present"] += 1

            if finite.any():
                target["global_min"] = min(
                    target["global_min"],
                    float(np.min(array[finite])),
                )
                target["global_max"] = max(
                    target["global_max"],
                    float(np.max(array[finite])),
                )

    except Exception as exc:
        schema_error_rows.append(
            {
                "symbol": symbol,
                "file_name": path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 25 == 0
        or file_number == len(train_file_map)
    ):
        elapsed = time.perf_counter() - started
        print(
            f"[feature quality] "
            f"[{file_number:>4}/{len(train_file_map)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(schema_error_rows)}"
        )

quality_rows = []

for feature, values in quality_accumulator.items():
    total_rows = int(values["total_rows"])
    finite_values = int(values["finite_values"])
    global_min = (
        float(values["global_min"])
        if np.isfinite(values["global_min"])
        else np.nan
    )
    global_max = (
        float(values["global_max"])
        if np.isfinite(values["global_max"])
        else np.nan
    )

    quality_rows.append(
        {
            "feature": feature,
            **values,
            "global_min": global_min,
            "global_max": global_max,
            "missing_fraction": (
                int(values["missing_values"]) / total_rows
                if total_rows
                else np.nan
            ),
            "nonfinite_fraction": (
                int(values["nonfinite_values"]) / total_rows
                if total_rows
                else np.nan
            ),
            "is_constant_on_finite_train_values": bool(
                finite_values > 0
                and np.isclose(global_min, global_max)
            ),
        }
    )

feature_quality_df = pd.DataFrame(quality_rows)
schema_errors_df = pd.DataFrame(
    schema_error_rows,
    columns=[
        "symbol",
        "file_name",
        "error_type",
        "error_message",
    ],
)

quality_config = feature_policy["data_quality"]

feature_approval_df = finalize_feature_approval(
    role_table_df,
    feature_quality_df,
    maximum_missing_fraction=float(
        quality_config["maximum_missing_fraction_for_approval"]
    ),
    reject_all_missing=bool(quality_config["reject_all_missing"]),
    reject_constant=bool(quality_config["reject_constant"]),
)

approved_model_features = feature_approval_df.loc[
    feature_approval_df["approved_for_modeling"],
    "feature",
].astype(str).tolist()

feature_quality_df.to_csv(
    RESULT_PATHS["audits"] / "04_feature_quality_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
feature_approval_df.to_csv(
    RESULT_PATHS["audits"] / "04_feature_approval_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
schema_errors_df.to_csv(
    RESULT_PATHS["audits"] / "04_feature_audit_errors.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)

approved_features_df = pd.DataFrame(
    {
        "feature_order": range(1, len(approved_model_features) + 1),
        "feature": approved_model_features,
    }
)
approved_features_df.to_csv(
    RESULT_PATHS["manifests"] / "04_approved_model_features.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)

print("Approved model features:", len(approved_model_features))
display(
    feature_approval_df[
        [
            "feature",
            "structural_role",
            "missing_fraction",
            "is_constant_on_finite_train_values",
            "approved_for_modeling",
            "approval_reason",
        ]
    ]
)


[feature quality] [   1/499] elapsed=0.0s errors=0
[feature quality] [  25/499] elapsed=0.9s errors=0
[feature quality] [  50/499] elapsed=1.7s errors=0
[feature quality] [  75/499] elapsed=2.7s errors=0
[feature quality] [ 100/499] elapsed=3.6s errors=0
[feature quality] [ 125/499] elapsed=4.4s errors=0
[feature quality] [ 150/499] elapsed=5.5s errors=0
[feature quality] [ 175/499] elapsed=6.6s errors=0
[feature quality] [ 200/499] elapsed=7.7s errors=0
[feature quality] [ 225/499] elapsed=8.8s errors=0
[feature quality] [ 250/499] elapsed=9.7s errors=0
[feature quality] [ 275/499] elapsed=10.7s errors=0
[feature quality] [ 300/499] elapsed=11.7s errors=0
[feature quality] [ 325/499] elapsed=12.9s errors=0
[feature quality] [ 350/499] elapsed=14.0s errors=0
[feature quality] [ 375/499] elapsed=14.9s errors=0
[feature quality] [ 400/499] elapsed=15.9s errors=0
[feature quality] [ 425/499] elapsed=16.9s errors=0
[feature quality] [ 450/499] elapsed=18.0s errors=0
[feature quality] [ 475

,feature,structural_role,missing_fraction,is_constant_on_finite_train_values,approved_for_modeling,approval_reason
0,adj_open,context_only,0.000000,False,False,context_only
1,adj_high,context_only,0.000000,False,False,context_only
2,adj_low,context_only,0.000000,False,False,context_only
3,adj_last_price,context_only,0.000000,False,False,context_only
4,zigzag_up_new_2,legacy_zigzag_rejected,0.012292,False,False,legacy_zigzag_rejected
5,zigzag_down_new_2,legacy_zigzag_rejected,0.012292,False,False,legacy_zigzag_rejected
6,EMA_3,model_candidate,0.001543,False,True,approved
7,EMA_5,model_candidate,0.003087,False,True,approved
8,EMA_8,model_candidate,0.005402,False,True,approved
9,EMA_10,model_candidate,0.006945,False,True,approved


## 6. Audit the supplied legacy ZigZag timing logic

The supplied collection code contains explicit pivot confirmation indices. The
legacy `new_2` distance loop, however, scans the pivot marker and skips the first
pivot rather than directly requiring the confirmation marker.

Stage 04 preserves the original **confirmation intent** and the original
`depth=10`, `deviation=15%` parameters, but replaces the skip-first heuristic
with a direct confirmation-time gate.


In [7]:
legacy_zigzag_logic_df = legacy_zigzag_source_logic_audit()

legacy_zigzag_logic_df.to_csv(
    RESULT_PATHS["audits"] / "04_legacy_zigzag_source_logic_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)

display(legacy_zigzag_logic_df)


,check,observed,risk,stage_04_action
0,confirmation_index_is_computed,True,False,retain design intent
1,confirmation_markers_are_created,True,False,retain design intent
2,legacy_new2_loop_scans_pivot_marker_zj,True,True,do not use legacy new_2 columns directly
3,legacy_new2_loop_explicitly_requires_confirmat...,False,True,reconstruct confirmation-gated state
4,legacy_new2_loop_skips_first_pivot,True,True,replace heuristic with confirmation timestamp ...
5,stage04_state_requires_confirmation_index_at_o...,True,False,required
6,stage04_state_is_prefix_invariance_audited,True,False,required


## 7. Reconstruct confirmation-gated ZigZag state and audit prefix invariance

Prefix invariance means that the state recorded for date `t` is identical whether
we calculate on data ending at `t` or on the full train history. This directly
tests the absence of historical repainting in the Stage 04 reconstruction.


In [8]:
rng = np.random.default_rng(1729)

sample_symbol_count = min(12, len(frozen_symbols))
prefix_positions_per_symbol = 12

sampled_symbols = sorted(
    rng.choice(
        np.asarray(frozen_symbols, dtype=object),
        size=sample_symbol_count,
        replace=False,
    ).tolist()
)

prefix_audit_rows = []

for symbol in sampled_symbols:
    path = train_file_map[symbol]
    frame = pd.read_csv(
        path,
        low_memory=False,
        usecols=[
            DATE_COLUMN,
            "adj_high",
            "adj_low",
            "adj_last_price",
        ],
    )
    frame[DATE_COLUMN] = pd.to_datetime(frame[DATE_COLUMN], errors="coerce")
    frame = frame.sort_values(DATE_COLUMN, kind="stable").reset_index(drop=True)

    minimum_position = max(
        64,
        zigzag_config.depth * 4,
    )

    if len(frame) <= minimum_position:
        positions = []
    else:
        candidate_positions = np.arange(
            minimum_position,
            len(frame),
            dtype=int,
        )
        sample_size = min(
            prefix_positions_per_symbol,
            len(candidate_positions),
        )
        positions = sorted(
            rng.choice(
                candidate_positions,
                size=sample_size,
                replace=False,
            ).tolist()
        )

    symbol_audit = prefix_invariance_audit(
        frame,
        config=zigzag_config,
        positions=positions,
        date_column=DATE_COLUMN,
    )
    symbol_audit.insert(0, "symbol", symbol)
    prefix_audit_rows.append(symbol_audit)

prefix_invariance_df = pd.concat(
    prefix_audit_rows,
    ignore_index=True,
) if prefix_audit_rows else pd.DataFrame(
    columns=[
        "symbol",
        "position",
        "event_date",
        "prefix_invariant",
        "mismatch_columns",
    ]
)

prefix_invariance_df.to_csv(
    RESULT_PATHS["audits"] / "04_zigzag_prefix_invariance_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)

prefix_mismatches = int(
    (~prefix_invariance_df["prefix_invariant"].astype(bool)).sum()
)

print("Prefix checks:", len(prefix_invariance_df))
print("Prefix mismatches:", prefix_mismatches)
display(prefix_invariance_df.head(20))


Prefix checks: 144
Prefix mismatches: 0


,symbol,position,event_date,prefix_invariant,mismatch_columns
0,بالبر,255,2015-11-02,True,
1,بالبر,290,2016-01-12,True,
2,بالبر,345,2016-04-06,True,
3,بالبر,434,2016-08-30,True,
4,بالبر,500,2016-12-25,True,
5,بالبر,575,2017-05-14,True,
6,بالبر,587,2017-05-30,True,
7,بالبر,628,2017-08-01,True,
8,بالبر,782,2018-03-27,True,
9,بالبر,1028,2019-04-22,True,


## 8. Train-only candidate-filter sensitivity and primary candidate generation

The 15% threshold is **not selected from these results**. It was already recorded
as the intended Stage 03 filter threshold.

The 10% and 20% rules are reported only to show how event density and label
composition change on train.


In [9]:
for old_file in CANDIDATE_TRAIN_PATH.glob("*_train_candidates.csv"):
    old_file.unlink()

threshold_accumulators = {
    threshold: defaultdict(float)
    for threshold in ALL_THRESHOLDS
}
threshold_year_accumulators = {
    threshold: defaultdict(
        lambda: defaultdict(float)
    )
    for threshold in ALL_THRESHOLDS
}

candidate_audit_rows = []
zigzag_state_audit_rows = []
candidate_write_errors = []

started = time.perf_counter()

for file_number, (symbol, path) in enumerate(
    sorted(train_file_map.items()),
    start=1,
):
    try:
        frame = pd.read_csv(path, low_memory=False)
        frame[DATE_COLUMN] = pd.to_datetime(
            frame[DATE_COLUMN],
            errors="coerce",
        )
        frame = frame.sort_values(
            DATE_COLUMN,
            kind="stable",
        ).reset_index(drop=True)

        state = build_confirmation_gated_zigzag_state(
            frame[
                [
                    DATE_COLUMN,
                    "adj_high",
                    "adj_low",
                    "adj_last_price",
                ]
            ],
            config=zigzag_config,
            date_column=DATE_COLUMN,
        )

        assert len(state) == len(frame)
        assert state[DATE_COLUMN].equals(frame[DATE_COLUMN])

        high_confirmation_after_row = (
            state["confirmed_zigzag_high_confirmation_date"].notna()
            & (
                pd.to_datetime(
                    state["confirmed_zigzag_high_confirmation_date"],
                    errors="coerce",
                )
                > frame[DATE_COLUMN]
            )
        )
        low_confirmation_after_row = (
            state["confirmed_zigzag_low_confirmation_date"].notna()
            & (
                pd.to_datetime(
                    state["confirmed_zigzag_low_confirmation_date"],
                    errors="coerce",
                )
                > frame[DATE_COLUMN]
            )
        )

        zigzag_state_audit_rows.append(
            {
                "symbol": symbol,
                "rows": len(frame),
                "state_available_rows": int(
                    state["confirmed_zigzag_state_available"].sum()
                ),
                "high_confirmation_after_event_rows": int(
                    high_confirmation_after_row.sum()
                ),
                "low_confirmation_after_event_rows": int(
                    low_confirmation_after_row.sum()
                ),
                "legacy_up_finite_rows": int(
                    np.isfinite(
                        pd.to_numeric(
                            frame["zigzag_up_new_2"],
                            errors="coerce",
                        ).to_numpy(dtype=float)
                    ).sum()
                ),
                "legacy_down_finite_rows": int(
                    np.isfinite(
                        pd.to_numeric(
                            frame["zigzag_down_new_2"],
                            errors="coerce",
                        ).to_numpy(dtype=float)
                    ).sum()
                ),
            }
        )

        eligible = frame["eligible_for_modeling"].fillna(False).astype(bool)

        for threshold in ALL_THRESHOLDS:
            candidate_mask = build_candidate_long_mask(
                state,
                eligible_for_modeling=eligible,
                threshold_fraction=threshold,
            )

            selected = frame.loc[candidate_mask]
            accumulator = threshold_accumulators[threshold]

            accumulator["rows"] += len(frame)
            accumulator["eligible_events"] += int(eligible.sum())
            accumulator["candidate_events"] += int(candidate_mask.sum())
            accumulator["positive_labels"] += int(
                selected["label"].eq(1).sum()
            )
            accumulator["negative_labels"] += int(
                selected["label"].eq(0).sum()
            )
            accumulator["upper_events"] += int(
                selected["barrier_touched"].eq("upper").sum()
            )
            accumulator["lower_events"] += int(
                selected["barrier_touched"].eq("lower").sum()
            )
            accumulator["vertical_events"] += int(
                selected["barrier_touched"].eq("vertical").sum()
            )
            accumulator["event_return_sum"] += float(
                pd.to_numeric(
                    selected["event_return"],
                    errors="coerce",
                ).sum()
            )
            accumulator["event_return_count"] += int(
                pd.to_numeric(
                    selected["event_return"],
                    errors="coerce",
                ).notna().sum()
            )

            if len(selected):
                selected_years = pd.to_datetime(
                    selected[DATE_COLUMN],
                    errors="coerce",
                ).dt.year

                for year, year_index in selected_years.groupby(
                    selected_years
                ).groups.items():
                    if pd.isna(year):
                        continue
                    year_selected = selected.loc[year_index]
                    year_target = threshold_year_accumulators[
                        threshold
                    ][int(year)]
                    year_target["candidate_events"] += len(year_selected)
                    year_target["positive_labels"] += int(
                        year_selected["label"].eq(1).sum()
                    )
                    year_target["negative_labels"] += int(
                        year_selected["label"].eq(0).sum()
                    )

        primary_mask = build_candidate_long_mask(
            state,
            eligible_for_modeling=eligible,
            threshold_fraction=PRIMARY_THRESHOLD,
        )

        primary_candidates = frame.loc[primary_mask].copy()
        primary_state = state.loc[primary_mask].reset_index(drop=True)

        state_columns_to_attach = [
            "confirmed_zigzag_high_price",
            "confirmed_zigzag_low_price",
            "confirmed_zigzag_high_pivot_date",
            "confirmed_zigzag_low_pivot_date",
            "confirmed_zigzag_high_confirmation_date",
            "confirmed_zigzag_low_confirmation_date",
            "distance_above_confirmed_low_fraction",
            "signed_distance_from_confirmed_high_fraction",
            "distance_below_confirmed_high_fraction",
            "confirmed_zigzag_state_available",
        ]

        primary_candidates = primary_candidates.reset_index(drop=True)
        primary_candidates[state_columns_to_attach] = primary_state[
            state_columns_to_attach
        ]

        primary_candidates["primary_side"] = 1
        primary_candidates["meta_label"] = pd.to_numeric(
            primary_candidates["label"],
            errors="coerce",
        ).astype("Int64")
        primary_candidates["candidate_filter_threshold_fraction"] = (
            PRIMARY_THRESHOLD
        )

        output_path = (
            CANDIDATE_TRAIN_PATH
            / f"{symbol}_train_candidates.csv"
        )
        primary_candidates.to_csv(
            output_path,
            index=False,
            encoding=OUTPUT_ENCODING,
        )

        candidate_dates = pd.to_datetime(
            primary_candidates[DATE_COLUMN],
            errors="coerce",
        ).sort_values()
        calendar_gaps = candidate_dates.diff().dt.days.dropna()

        candidate_audit_rows.append(
            {
                "symbol": symbol,
                "eligible_train_events": int(eligible.sum()),
                "primary_candidate_events": int(primary_mask.sum()),
                "candidate_fraction_of_eligible": (
                    float(primary_mask.sum() / eligible.sum())
                    if int(eligible.sum())
                    else np.nan
                ),
                "positive_meta_labels": int(
                    primary_candidates["meta_label"].eq(1).sum()
                ),
                "negative_meta_labels": int(
                    primary_candidates["meta_label"].eq(0).sum()
                ),
                "positive_meta_label_fraction": (
                    float(primary_candidates["meta_label"].eq(1).mean())
                    if len(primary_candidates)
                    else np.nan
                ),
                "median_calendar_gap_between_candidates": (
                    float(calendar_gaps.median())
                    if len(calendar_gaps)
                    else np.nan
                ),
                "first_candidate_date": (
                    candidate_dates.min()
                    if len(candidate_dates)
                    else pd.NaT
                ),
                "last_candidate_date": (
                    candidate_dates.max()
                    if len(candidate_dates)
                    else pd.NaT
                ),
                "output_rows": len(primary_candidates),
                "output_file": output_path.name,
            }
        )

    except Exception as exc:
        candidate_write_errors.append(
            {
                "symbol": symbol,
                "file_name": path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 25 == 0
        or file_number == len(train_file_map)
    ):
        elapsed = time.perf_counter() - started
        print(
            f"[ZigZag/candidate] "
            f"[{file_number:>4}/{len(train_file_map)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(candidate_write_errors)}"
        )

threshold_rows = []

for threshold in ALL_THRESHOLDS:
    values = threshold_accumulators[threshold]
    candidate_events = int(values["candidate_events"])
    eligible_events = int(values["eligible_events"])

    threshold_rows.append(
        {
            "threshold_fraction": threshold,
            "is_primary_pre_registered_threshold": np.isclose(
                threshold,
                PRIMARY_THRESHOLD,
            ),
            "comparison_scope": "train_only",
            "automatic_threshold_selection_used": False,
            **{
                key: int(value)
                if key not in {
                    "event_return_sum",
                }
                else float(value)
                for key, value in values.items()
                if key != "event_return_count"
            },
            "candidate_fraction_of_eligible": (
                candidate_events / eligible_events
                if eligible_events
                else np.nan
            ),
            "positive_meta_label_fraction": (
                values["positive_labels"] / candidate_events
                if candidate_events
                else np.nan
            ),
            "mean_event_return": (
                values["event_return_sum"]
                / values["event_return_count"]
                if values["event_return_count"]
                else np.nan
            ),
        }
    )

threshold_comparison_df = pd.DataFrame(threshold_rows)

threshold_year_rows = []
for threshold in ALL_THRESHOLDS:
    for year, values in sorted(
        threshold_year_accumulators[threshold].items()
    ):
        candidate_events = int(values["candidate_events"])
        threshold_year_rows.append(
            {
                "threshold_fraction": threshold,
                "calendar_year": int(year),
                **{
                    key: int(value)
                    for key, value in values.items()
                },
                "positive_meta_label_fraction": (
                    values["positive_labels"] / candidate_events
                    if candidate_events
                    else np.nan
                ),
            }
        )

threshold_by_year_df = pd.DataFrame(threshold_year_rows)
candidate_audit_df = pd.DataFrame(candidate_audit_rows)
zigzag_state_audit_df = pd.DataFrame(zigzag_state_audit_rows)
candidate_errors_df = pd.DataFrame(
    candidate_write_errors,
    columns=[
        "symbol",
        "file_name",
        "error_type",
        "error_message",
    ],
)

threshold_comparison_df.to_csv(
    RESULT_PATHS["audits"] / "04_candidate_filter_threshold_comparison.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
threshold_by_year_df.to_csv(
    RESULT_PATHS["audits"] / "04_candidate_filter_by_year.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
candidate_audit_df.to_csv(
    RESULT_PATHS["audits"] / "04_candidate_event_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
zigzag_state_audit_df.to_csv(
    RESULT_PATHS["audits"] / "04_confirmation_gated_zigzag_state_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
candidate_errors_df.to_csv(
    RESULT_PATHS["audits"] / "04_candidate_generation_errors.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)

display(threshold_comparison_df)


[ZigZag/candidate] [   1/499] elapsed=0.2s errors=0
[ZigZag/candidate] [  25/499] elapsed=4.1s errors=0
[ZigZag/candidate] [  50/499] elapsed=8.1s errors=0
[ZigZag/candidate] [  75/499] elapsed=12.1s errors=0
[ZigZag/candidate] [ 100/499] elapsed=16.3s errors=0
[ZigZag/candidate] [ 125/499] elapsed=20.6s errors=0
[ZigZag/candidate] [ 150/499] elapsed=24.5s errors=0
[ZigZag/candidate] [ 175/499] elapsed=28.9s errors=0
[ZigZag/candidate] [ 200/499] elapsed=32.5s errors=0
[ZigZag/candidate] [ 225/499] elapsed=36.2s errors=0
[ZigZag/candidate] [ 250/499] elapsed=40.2s errors=0
[ZigZag/candidate] [ 275/499] elapsed=43.9s errors=0
[ZigZag/candidate] [ 300/499] elapsed=47.7s errors=0
[ZigZag/candidate] [ 325/499] elapsed=51.6s errors=0
[ZigZag/candidate] [ 350/499] elapsed=55.5s errors=0
[ZigZag/candidate] [ 375/499] elapsed=59.6s errors=0
[ZigZag/candidate] [ 400/499] elapsed=63.4s errors=0
[ZigZag/candidate] [ 425/499] elapsed=67.0s errors=0
[ZigZag/candidate] [ 450/499] elapsed=70.6s error

,threshold_fraction,is_primary_pre_registered_threshold,comparison_scope,automatic_threshold_selection_used,rows,eligible_events,candidate_events,positive_labels,negative_labels,upper_events,lower_events,vertical_events,event_return_sum,candidate_fraction_of_eligible,positive_meta_label_fraction,mean_event_return
0,0.10,False,train_only,False,658777,646618,110814,59629,51185,40403,19000,51411,2366.532236,0.171375,0.538100,0.021356
1,0.15,True,train_only,False,658777,646618,118464,63873,54591,45031,21599,51834,2573.748497,0.183206,0.539176,0.021726
2,0.20,False,train_only,False,658777,646618,102362,56044,46318,41021,19553,41788,2424.352152,0.158304,0.547508,0.023684


## 9. Final pooled-model feature engineering

The candidate-event definition remains unchanged. This section reconstructs
cross-sectionally comparable stock features and adds nine causal
**equal-weight market-regime features**.

### Global market-calendar rule

The equal-weight index columns are repeated inside stock raw files, but a stock
file may omit dates when that security did not trade. Therefore market rolling
windows are not calculated separately inside each symbol file.

Notebook 04 pools the repeated index observations from all frozen-symbol raw
files, checks same-date consistency, creates one canonical global market
calendar, and calculates market-regime features once on that calendar.

### Feature provenance

The two confirmed ZigZag geometry variables remain carried directly from the
confirmation-gated Stage 04 candidate state.


In [10]:
import importlib
import sys

FEATURE_ENGINEERING_SCHEMA_VERSION_EXPECTED = (
    "stage04_pooled_v7_market_regime_features"
)

importlib.invalidate_caches()
sys.modules.pop("src.features.preprocessing", None)

preprocessing_module = importlib.import_module(
    "src.features.preprocessing"
)

print("preprocessing.py loaded from:", preprocessing_module.__file__)
print(
    "Feature schema version:",
    getattr(
        preprocessing_module,
        "FEATURE_ENGINEERING_SCHEMA_VERSION",
        "<missing>",
    ),
)

assert (
    getattr(
        preprocessing_module,
        "FEATURE_ENGINEERING_SCHEMA_VERSION",
        None,
    )
    == FEATURE_ENGINEERING_SCHEMA_VERSION_EXPECTED
), (
    "Wrong src/features/preprocessing.py is loaded. "
    "Replace the repository file with the Stage 04 v7 patch."
)

from src.features.preprocessing import (
    CARRIED_STAGE04_NUMERIC_FEATURES,
    ENGINEERED_MODEL_FEATURES,
    ENGINEERED_NUMERIC_FEATURES,
    FINAL_CATEGORICAL_FEATURES,
    FINAL_MODEL_FEATURES,
    FINAL_NUMERIC_FEATURES,
    MARKET_INDEX_REQUIRED_COLUMNS,
    MARKET_REGIME_NUMERIC_FEATURES,
    FeatureEngineeringConfig,
    build_canonical_market_index,
    build_final_feature_frame,
    build_market_regime_feature_frame,
    final_feature_schema,
    parse_market_date,
    prepare_market_index_source,
)

RAW_DATA_PATH = DATA_PATHS["raw_data"]

raw_files = sorted(RAW_DATA_PATH.glob("*.csv"))
raw_file_map = {path.stem: path for path in raw_files}

assert set(raw_file_map).issuperset(frozen_symbol_set), (
    "At least one frozen symbol has no matching raw-data CSV."
)

final_feature_config = FeatureEngineeringConfig(
    relative_strength_window=12,
    return_zscore_window=12,
    volume_window=30,
    market_volatility_window=20,
    market_ema_fast_window=20,
    market_ema_slow_window=60,
    market_drawdown_window=60,
    market_index_consistency_relative_tolerance=1e-10,
)

final_feature_schema_df = final_feature_schema()

assert len(MARKET_REGIME_NUMERIC_FEATURES) == 9
assert len(ENGINEERED_NUMERIC_FEATURES) == 32
assert len(CARRIED_STAGE04_NUMERIC_FEATURES) == 2
assert len(FINAL_NUMERIC_FEATURES) == 34
assert len(FINAL_CATEGORICAL_FEATURES) == 1
assert len(ENGINEERED_MODEL_FEATURES) == 33
assert len(FINAL_MODEL_FEATURES) == 35
assert tuple(FINAL_CATEGORICAL_FEATURES) == ("gmma_state",)

display(final_feature_schema_df)


preprocessing.py loaded from: E:\Iran_stock_trade\financial-ml-decision-support\src\features\preprocessing.py
Feature schema version: stage04_pooled_v7_market_regime_features


,feature_order,feature,semantic_group,source_feature,transformation,unit_before,unit_after,data_type,price_basis,approved_for_pooled_model
0,1,ema_3_distance,trend_location,EMA_3,(adj_last_price - EMA_3) / adj_last_price,price,dimensionless,numeric,adjusted,True
1,2,ema_5_distance,trend_location,EMA_5,(adj_last_price - EMA_5) / adj_last_price,price,dimensionless,numeric,adjusted,True
2,3,ema_8_distance,trend_location,EMA_8,(adj_last_price - EMA_8) / adj_last_price,price,dimensionless,numeric,adjusted,True
3,4,ema_10_distance,trend_location,EMA_10,(adj_last_price - EMA_10) / adj_last_price,price,dimensionless,numeric,adjusted,True
4,5,ema_12_distance,trend_location,EMA_12,(adj_last_price - EMA_12) / adj_last_price,price,dimensionless,numeric,adjusted,True
5,6,ema_15_distance,trend_location,EMA_15,(adj_last_price - EMA_15) / adj_last_price,price,dimensionless,numeric,adjusted,True
6,7,ema_30_distance,trend_location,EMA_30,(adj_last_price - EMA_30) / adj_last_price,price,dimensionless,numeric,adjusted,True
7,8,ema_35_distance,trend_location,EMA_35,(adj_last_price - EMA_35) / adj_last_price,price,dimensionless,numeric,adjusted,True
8,9,ema_40_distance,trend_location,EMA_40,(adj_last_price - EMA_40) / adj_last_price,price,dimensionless,numeric,adjusted,True
9,10,ema_45_distance,trend_location,EMA_45,(adj_last_price - EMA_45) / adj_last_price,price,dimensionless,numeric,adjusted,True


### 9.1 Build one canonical equal-weight market-index calendar

Source columns:

- `xNivInuClMresIbs`: equal-weight index close/current level
- `xNivInuPbMresIbs`: equal-weight index daily low
- `xNivInuPhMresIbs`: equal-weight index daily high

For an ordinary day:

`market_close_location = (close - low) / (high - low)`

For `high == low == close`:

- current close > previous market close → `1`
- current close < previous market close → `0`
- current close == previous market close → missing


### Train-horizon scope

The canonical market calendar used in Stage 04 is restricted to the maximum
date actually present in the frozen Stage 03 labeled-train files.

Post-train raw index observations are excluded **before** cross-file
consistency checks and before any rolling market feature is calculated.

This keeps Stage 04 feature approval aligned with its train-only scope and
prevents out-of-scope raw revisions or snapshot differences from stopping the
train-only audit.


In [11]:
# Determine the actual final date available in the frozen labeled-train inputs.
train_horizon_end_dates = []

for labeled_path in train_file_map.values():
    labeled_dates = pd.read_csv(
        labeled_path,
        usecols=[DATE_COLUMN],
        low_memory=False,
    )
    parsed_labeled_dates = pd.to_datetime(
        labeled_dates[DATE_COLUMN],
        errors="coerce",
    )

    if parsed_labeled_dates.notna().any():
        train_horizon_end_dates.append(
            parsed_labeled_dates.max()
        )

if not train_horizon_end_dates:
    raise RuntimeError(
        "No valid labeled-train date was available to freeze "
        "the Stage 04 market feature horizon."
    )

TRAIN_FEATURE_HORIZON_END = max(train_horizon_end_dates)

assert TRAIN_FEATURE_HORIZON_END <= pd.Timestamp("2021-03-20"), (
    "The labeled-train feature horizon exceeds the frozen nominal "
    "train cutoff."
)

market_index_parts = []
market_index_error_rows = []
market_rows_after_train_horizon = 0
market_rows_invalid_date = 0

started = time.perf_counter()

for file_number, (symbol, raw_path) in enumerate(
    sorted(raw_file_map.items()),
    start=1,
):
    if symbol not in frozen_symbol_set:
        continue

    try:
        raw_market_frame = pd.read_csv(
            raw_path,
            usecols=list(MARKET_INDEX_REQUIRED_COLUMNS),
            low_memory=False,
        )

        raw_market_dates = parse_market_date(
            raw_market_frame[DATE_COLUMN]
        )

        market_rows_invalid_date += int(
            raw_market_dates.isna().sum()
        )
        market_rows_after_train_horizon += int(
            raw_market_dates.gt(
                TRAIN_FEATURE_HORIZON_END
            ).sum()
        )

        in_train_feature_horizon = (
            raw_market_dates.notna()
            & raw_market_dates.le(
                TRAIN_FEATURE_HORIZON_END
            )
        )

        raw_market_frame = raw_market_frame.loc[
            in_train_feature_horizon
        ].copy()

        market_index_parts.append(
            prepare_market_index_source(
                raw_market_frame,
                source_symbol=symbol,
            )
        )

    except Exception as exc:
        market_index_error_rows.append(
            {
                "symbol": symbol,
                "file_name": raw_path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 50 == 0
        or file_number == len(raw_file_map)
    ):
        elapsed = time.perf_counter() - started
        print(
            f"[market index] "
            f"[{file_number:>4}/{len(raw_file_map)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(market_index_error_rows)}"
        )

market_index_errors_df = pd.DataFrame(
    market_index_error_rows,
    columns=[
        "symbol",
        "file_name",
        "error_type",
        "error_message",
    ],
)

if not market_index_errors_df.empty:
    market_index_errors_df.to_csv(
        RESULT_PATHS["audits"]
        / "04_market_index_source_errors.csv",
        index=False,
        encoding=OUTPUT_ENCODING,
    )
    raise RuntimeError(
        "Market-index source errors exist. "
        "Review 04_market_index_source_errors.csv"
    )

market_observation_panel = pd.concat(
    market_index_parts,
    ignore_index=True,
)

assert market_observation_panel["dEven"].max() <= (
    TRAIN_FEATURE_HORIZON_END
)

canonical_market_index_df, market_index_consistency_audit_df = (
    build_canonical_market_index(
        market_observation_panel,
        relative_tolerance=(
            final_feature_config
            .market_index_consistency_relative_tolerance
        ),
    )
)

market_regime_feature_frame, market_regime_audit = (
    build_market_regime_feature_frame(
        canonical_market_index_df,
        config=final_feature_config,
    )
)

market_regime_audit = {
    **market_regime_audit,
    "train_feature_horizon_end": (
        TRAIN_FEATURE_HORIZON_END
    ),
    "raw_market_rows_excluded_after_train_horizon": (
        market_rows_after_train_horizon
    ),
    "raw_market_rows_invalid_date": (
        market_rows_invalid_date
    ),
}

market_regime_audit_df = pd.DataFrame(
    [market_regime_audit]
)

market_index_consistency_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "04_market_index_consistency_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
market_regime_audit_df.to_csv(
    RESULT_PATHS["audits"]
    / "04_market_regime_feature_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)

inconsistent_market_rows = int(
    market_index_consistency_audit_df[
        "inconsistent_across_raw_files"
    ].sum()
)

assert canonical_market_index_df["dEven"].max() <= (
    TRAIN_FEATURE_HORIZON_END
)
assert market_regime_feature_frame["dEven"].max() <= (
    TRAIN_FEATURE_HORIZON_END
)

assert inconsistent_market_rows == 0, (
    "Equal-weight index values disagree across frozen-symbol raw files "
    "inside the train feature horizon. "
    "Review 04_market_index_consistency_audit.csv"
)

assert market_regime_feature_frame["dEven"].is_unique
assert market_regime_feature_frame["dEven"].is_monotonic_increasing

print(
    "Stage 04 market feature horizon end:",
    TRAIN_FEATURE_HORIZON_END.date(),
)
print(
    "Raw market rows excluded after train horizon:",
    market_rows_after_train_horizon,
)
print(
    "Canonical market trading dates:",
    len(canonical_market_index_df),
)
print(
    "Market-index inconsistencies inside train horizon:",
    inconsistent_market_rows,
)

display(market_regime_audit_df)
display(
    market_regime_feature_frame[
        ["dEven", *MARKET_REGIME_NUMERIC_FEATURES]
    ].tail(10)
)


[market index] [   1/825] elapsed=0.4s errors=0
[market index] [ 200/825] elapsed=8.0s errors=0
[market index] [ 250/825] elapsed=10.9s errors=0
[market index] [ 300/825] elapsed=12.7s errors=0
[market index] [ 350/825] elapsed=14.4s errors=0
[market index] [ 400/825] elapsed=16.6s errors=0
[market index] [ 450/825] elapsed=18.2s errors=0
[market index] [ 500/825] elapsed=19.8s errors=0
[market index] [ 550/825] elapsed=22.0s errors=0
[market index] [ 600/825] elapsed=23.8s errors=0
[market index] [ 700/825] elapsed=27.4s errors=0
[market index] [ 750/825] elapsed=28.5s errors=0
[market index] [ 800/825] elapsed=30.8s errors=0
[market index] [ 825/825] elapsed=31.7s errors=0
Stage 04 market feature horizon end: 2021-03-17
Raw market rows excluded after train horizon: 537101
Canonical market trading dates: 1690
Market-index inconsistencies inside train horizon: 0


,market_calendar_rows,market_first_date,market_last_date,invalid_market_ohlc_rows,locked_market_rows,locked_market_up_rows,locked_market_down_rows,locked_market_equal_previous_rows,market_close_location_missing_rows,market_return_1_missing_rows,market_return_5_missing_rows,market_return_20_missing_rows,market_volatility_20_missing_rows,market_ema_20_distance_missing_rows,market_ema_60_distance_missing_rows,market_range_fraction_missing_rows,market_drawdown_60_missing_rows,train_feature_horizon_end,raw_market_rows_excluded_after_train_horizon,raw_market_rows_invalid_date
0,1690,2014-03-19,2021-03-17,8,228,113,114,0,9,1,5,20,20,19,59,8,59,2021-03-17,537101,0


,dEven,market_return_1,market_return_5,market_return_20,market_volatility_20,market_ema_20_distance,market_ema_60_distance,market_range_fraction,market_close_location,market_drawdown_60
1680,2021-03-06,-0.000744,-0.005671,0.018421,0.008327,-0.005668,-0.006712,0.001172,0.017822,-0.089731
1681,2021-03-07,-0.000200,-0.003548,0.035754,0.007246,-0.005309,-0.006686,0.000845,0.219780,-0.089913
1682,2021-03-08,0.002769,0.001449,0.026323,0.006870,-0.002292,-0.003778,0.002363,0.953967,-0.087393
1683,2021-03-09,0.000690,0.001659,0.029172,0.006828,-0.001449,-0.002985,0.001670,0.077562,-0.086764
1684,2021-03-10,-0.001214,0.001296,0.020716,0.006726,-0.002412,-0.004067,0.000845,0.082192,-0.087873
1685,2021-03-13,0.003325,0.005374,-0.005429,0.001983,0.000823,-0.000715,0.003277,0.955634,-0.084839
1686,2021-03-14,0.002382,0.007970,-0.006431,0.001901,0.002893,0.001609,0.002091,0.712555,-0.082660
1687,2021-03-15,0.001262,0.006455,-0.005471,0.001928,0.003754,0.002773,0.002033,0.891403,-0.081502
1688,2021-03-16,0.007113,0.012916,-0.000233,0.002477,0.009763,0.009494,0.006903,1.000000,-0.074969
1689,2021-03-17,0.004268,0.018475,0.001923,0.002610,0.012641,0.013254,0.002719,0.984950,-0.071021


In [12]:
final_feature_audit_rows = []
final_feature_error_rows = []
final_feature_quality_parts = []

started = time.perf_counter()

for file_number, (symbol, labeled_path) in enumerate(
    sorted(train_file_map.items()),
    start=1,
):
    try:
        candidate_path = (
            CANDIDATE_TRAIN_PATH
            / f"{symbol}_train_candidates.csv"
        )
        raw_path = raw_file_map[symbol]

        labeled_frame = pd.read_csv(
            labeled_path,
            low_memory=False,
        )
        raw_frame = pd.read_csv(
            raw_path,
            low_memory=False,
        )
        candidate_frame = pd.read_csv(
            candidate_path,
            low_memory=False,
        )

        final_feature_frame, feature_audit = build_final_feature_frame(
            labeled_frame,
            raw_frame,
            market_regime_feature_frame,
            config=final_feature_config,
        )

        final_feature_frame["dEven"] = pd.to_datetime(
            final_feature_frame["dEven"],
            errors="coerce",
        )
        candidate_frame["dEven"] = pd.to_datetime(
            candidate_frame["dEven"],
            errors="coerce",
        )

        missing_carried_zigzag_columns = sorted(
            set(CARRIED_STAGE04_NUMERIC_FEATURES)
            - set(candidate_frame.columns)
        )
        if missing_carried_zigzag_columns:
            raise KeyError(
                "Stage 04 candidate file is missing frozen confirmed-ZigZag "
                f"geometry columns: {missing_carried_zigzag_columns}"
            )

        # Re-running Notebook 04 may find previously engineered columns in the
        # candidate CSV. Replace only deterministic engineered features.
        # The two confirmed-ZigZag distance columns are Stage 04 state carried
        # from candidate generation and must never be dropped or reconstructed
        # by build_final_feature_frame().
        existing_engineered_columns = [
            column
            for column in ENGINEERED_MODEL_FEATURES
            if column in candidate_frame.columns
        ]
        if existing_engineered_columns:
            candidate_frame = candidate_frame.drop(
                columns=existing_engineered_columns
            )

        missing_engineered_columns = sorted(
            set(ENGINEERED_MODEL_FEATURES)
            - set(final_feature_frame.columns)
        )
        if missing_engineered_columns:
            raise KeyError(
                "Engineered feature frame is missing expected columns: "
                f"{missing_engineered_columns}"
            )

        enriched_candidates = candidate_frame.merge(
            final_feature_frame[
                ["dEven", *ENGINEERED_MODEL_FEATURES]
            ],
            on="dEven",
            how="left",
            validate="one_to_one",
        )

        missing_final_columns = sorted(
            set(FINAL_MODEL_FEATURES)
            - set(enriched_candidates.columns)
        )
        if missing_final_columns:
            raise KeyError(
                "Enriched candidate file is missing final model columns: "
                f"{missing_final_columns}"
            )

        enriched_candidates.to_csv(
            candidate_path,
            index=False,
            encoding=OUTPUT_ENCODING,
        )

        candidate_feature_view = enriched_candidates[
            list(FINAL_MODEL_FEATURES)
        ].copy()

        per_feature_quality = []
        for feature in FINAL_NUMERIC_FEATURES:
            values = pd.to_numeric(
                candidate_feature_view[feature],
                errors="coerce",
            )
            array = values.to_numpy(dtype=float)
            finite = np.isfinite(array)

            per_feature_quality.append(
                {
                    "symbol": symbol,
                    "feature": feature,
                    "rows": len(values),
                    "missing_values": int(values.isna().sum()),
                    "nonfinite_values": int(np.isinf(array).sum()),
                    "finite_values": int(finite.sum()),
                    "minimum": (
                        float(array[finite].min())
                        if finite.any()
                        else np.nan
                    ),
                    "maximum": (
                        float(array[finite].max())
                        if finite.any()
                        else np.nan
                    ),
                }
            )

        gmma_values = candidate_feature_view["gmma_state"].astype("string")
        per_feature_quality.append(
            {
                "symbol": symbol,
                "feature": "gmma_state",
                "rows": len(gmma_values),
                "missing_values": int(gmma_values.isna().sum()),
                "nonfinite_values": 0,
                "finite_values": int(gmma_values.notna().sum()),
                "minimum": np.nan,
                "maximum": np.nan,
            }
        )

        final_feature_quality_parts.append(
            pd.DataFrame(per_feature_quality)
        )

        final_feature_audit_rows.append(
            {
                "symbol": symbol,
                "candidate_rows": len(enriched_candidates),
                "candidate_rows_with_all_final_features": int(
                    enriched_candidates[
                        list(FINAL_MODEL_FEATURES)
                    ].notna().all(axis=1).sum()
                ),
                "candidate_rows_with_any_final_feature_missing": int(
                    enriched_candidates[
                        list(FINAL_MODEL_FEATURES)
                    ].isna().any(axis=1).sum()
                ),
                **feature_audit,
            }
        )

    except Exception as exc:
        final_feature_error_rows.append(
            {
                "symbol": symbol,
                "file_name": labeled_path.name,
                "error_type": type(exc).__name__,
                "error_message": str(exc),
            }
        )

    if (
        file_number == 1
        or file_number % 25 == 0
        or file_number == len(train_file_map)
    ):
        elapsed = time.perf_counter() - started
        print(
            f"[final features] "
            f"[{file_number:>4}/{len(train_file_map)}] "
            f"elapsed={elapsed:,.1f}s "
            f"errors={len(final_feature_error_rows)}"
        )

final_feature_audit_df = pd.DataFrame(final_feature_audit_rows)
final_feature_errors_df = pd.DataFrame(
    final_feature_error_rows,
    columns=[
        "symbol",
        "file_name",
        "error_type",
        "error_message",
    ],
)

if not final_feature_quality_parts:
    final_feature_errors_df.to_csv(
        RESULT_PATHS["audits"]
        / "04_final_feature_engineering_errors.csv",
        index=False,
        encoding=OUTPUT_ENCODING,
    )

    first_errors = final_feature_errors_df.head(10).to_dict(
        orient="records"
    )
    raise RuntimeError(
        "Final feature engineering failed for every symbol. "
        "The original per-symbol errors were written to "
        "results/audits/04_final_feature_engineering_errors.csv. "
        f"First errors: {first_errors}"
    )

final_feature_quality_by_symbol_df = pd.concat(
    final_feature_quality_parts,
    ignore_index=True,
)

quality_rows = []

for feature in FINAL_MODEL_FEATURES:
    feature_rows = final_feature_quality_by_symbol_df.loc[
        final_feature_quality_by_symbol_df["feature"].eq(feature)
    ]

    total_rows = int(feature_rows["rows"].sum())
    missing_values = int(feature_rows["missing_values"].sum())
    nonfinite_values = int(feature_rows["nonfinite_values"].sum())
    finite_values = int(feature_rows["finite_values"].sum())

    quality_rows.append(
        {
            "feature": feature,
            "rows": total_rows,
            "missing_values": missing_values,
            "missing_fraction": (
                missing_values / total_rows
                if total_rows
                else np.nan
            ),
            "nonfinite_values": nonfinite_values,
            "finite_values": finite_values,
            "global_minimum": (
                feature_rows["minimum"].min()
                if feature != "gmma_state"
                else np.nan
            ),
            "global_maximum": (
                feature_rows["maximum"].max()
                if feature != "gmma_state"
                else np.nan
            ),
        }
    )

final_feature_quality_df = pd.DataFrame(quality_rows)

final_feature_schema_df.to_csv(
    RESULT_PATHS["manifests"] / "04_final_model_feature_schema.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
final_feature_audit_df.to_csv(
    RESULT_PATHS["audits"] / "04_final_feature_engineering_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
final_feature_quality_df.to_csv(
    RESULT_PATHS["audits"] / "04_final_feature_quality_audit.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)
final_feature_errors_df.to_csv(
    RESULT_PATHS["audits"] / "04_final_feature_engineering_errors.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)

approved_model_features = list(FINAL_MODEL_FEATURES)

pd.DataFrame(
    {
        "feature_order": range(
            1,
            len(approved_model_features) + 1,
        ),
        "feature": approved_model_features,
        "data_type": [
            (
                "categorical"
                if feature in FINAL_CATEGORICAL_FEATURES
                else "numeric"
            )
            for feature in approved_model_features
        ],
    }
).to_csv(
    RESULT_PATHS["manifests"] / "04_approved_model_features.csv",
    index=False,
    encoding=OUTPUT_ENCODING,
)

print("Final semantic model inputs:", len(approved_model_features))
print("Final numeric inputs:", len(FINAL_NUMERIC_FEATURES))
print("Final categorical inputs:", len(FINAL_CATEGORICAL_FEATURES))
display(final_feature_quality_df)


[final features] [   1/499] elapsed=0.4s errors=0
[final features] [  25/499] elapsed=7.6s errors=0
[final features] [  50/499] elapsed=15.0s errors=0
[final features] [  75/499] elapsed=22.7s errors=0
[final features] [ 100/499] elapsed=31.3s errors=0
[final features] [ 125/499] elapsed=39.8s errors=0
[final features] [ 150/499] elapsed=48.0s errors=0
[final features] [ 175/499] elapsed=62.3s errors=0
[final features] [ 200/499] elapsed=70.8s errors=0
[final features] [ 225/499] elapsed=79.8s errors=0
[final features] [ 250/499] elapsed=88.3s errors=0
[final features] [ 275/499] elapsed=97.6s errors=0
[final features] [ 300/499] elapsed=106.8s errors=0
[final features] [ 325/499] elapsed=116.4s errors=0
[final features] [ 350/499] elapsed=125.0s errors=0
[final features] [ 375/499] elapsed=134.1s errors=0
[final features] [ 400/499] elapsed=143.2s errors=0
[final features] [ 425/499] elapsed=151.9s errors=0
[final features] [ 450/499] elapsed=161.1s errors=0
[final features] [ 475/499

,feature,rows,missing_values,missing_fraction,nonfinite_values,finite_values,global_minimum,global_maximum
0,ema_3_distance,118464,0,0.000000,0,118464,-1.173345e-01,0.128040
1,ema_5_distance,118464,0,0.000000,0,118464,-1.668850e-01,0.175957
2,ema_8_distance,118464,0,0.000000,0,118464,-2.219899e-01,0.207525
3,ema_10_distance,118464,0,0.000000,0,118464,-2.688370e-01,0.223035
4,ema_12_distance,118464,0,0.000000,0,118464,-3.062067e-01,0.245137
5,ema_15_distance,118464,0,0.000000,0,118464,-3.479448e-01,0.269556
6,ema_30_distance,118464,1,0.000008,0,118463,-6.601250e-01,0.324495
7,ema_35_distance,118464,69,0.000582,0,118395,-7.344614e-01,0.329293
8,ema_40_distance,118464,192,0.001621,0,118272,-7.976501e-01,0.330422
9,ema_45_distance,118464,380,0.003208,0,118084,-8.476331e-01,0.329064


### Frozen semantic corrections

- Raw EMA levels are replaced by signed EMA-to-price distances.
- Raw MACD is divided by adjusted last price.
- RSI is centered around 50.
- Buyer power preserves the collection-pipeline `0 -> 1` continuity convention
  and is natural-log transformed.
- `ho_sell` uses total sell volume in its denominator.
- `x` uses the stock/equal-weight-index relative-strength ratio.
- `y` uses the standard return `(P_t / P_(t-1)) - 1`.
- `color` is removed.
- `started` is replaced by log positive-relative-strength run length.
- `gmma` is categorical.
- `body_ratio` retains the unadjusted-price Iranian market rule.
- Nine causal global equal-weight market-regime features are added.
- Confirmed ZigZag distances remain carried meta-model inputs.

### Market-regime feature set

- `market_return_1`
- `market_return_5`
- `market_return_20`
- `market_volatility_20`
- `market_ema_20_distance`
- `market_ema_60_distance`
- `market_range_fraction`
- `market_close_location`
- `market_drawdown_60`


## 10. Freeze Stage 04 manifests

In [13]:
candidate_policy_manifest = {
    "stage": 4,
    "status": "frozen_after_integrity_checks",
    "primary_side": "long",
    "primary_threshold_fraction": PRIMARY_THRESHOLD,
    "threshold_selection_scope": "train_only",
    "automatic_threshold_selection": False,
    "sensitivity_threshold_fractions": SENSITIVITY_THRESHOLDS,
    "low_distance_column": candidate_policy["low_distance_column"],
    "high_distance_column": candidate_policy["high_distance_column"],
    "legacy_zigzag_columns_used_directly": False,
    "confirmation_gated_reconstruction": True,
    "meta_label_column": "meta_label",
    "meta_label_source": "frozen Stage 03 Triple-Barrier label",
    "unseen_test_used": False,
}

candidate_policy_path = (
    RESULT_PATHS["manifests"]
    / "04_candidate_filter_policy.json"
)
candidate_policy_path.write_text(
    json.dumps(
        candidate_policy_manifest,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

run_manifest = {
    "stage": 4,
    "notebook": "04_feature_and_leakage_audit.ipynb",
    "git_commit_sha": git_commit_sha(REPOSITORY_ROOT),
    "software": software_manifest(),
    "frozen_universe_size": len(frozen_symbols),
    "approved_model_feature_count": len(approved_model_features),
    "approved_model_features": approved_model_features,
    "feature_approval_scope": "train_only",
    "feature_engineering_schema_version": (
        preprocessing_module.FEATURE_ENGINEERING_SCHEMA_VERSION
    ),
    "market_regime_feature_count": len(MARKET_REGIME_NUMERIC_FEATURES),
    "market_regime_features": list(MARKET_REGIME_NUMERIC_FEATURES),
    "market_index_calendar_source": "pooled frozen-symbol raw files",
    "market_feature_horizon_end": str(
        TRAIN_FEATURE_HORIZON_END.date()
    ),
    "market_feature_scope": "frozen_labeled_train_horizon_only",
    "post_train_market_values_used_for_feature_engineering": False,
    "market_index_cross_file_inconsistencies": inconsistent_market_rows,
    "unseen_test_used_for_feature_approval": False,
    "unseen_test_used_for_threshold_selection": False,
    "zigzag": {
        "depth": zigzag_config.depth,
        "deviation_percent": zigzag_config.deviation_percent,
        "legacy_new2_used_directly": False,
        "confirmation_gated_reconstruction": True,
        "prefix_checks": len(prefix_invariance_df),
        "prefix_mismatches": prefix_mismatches,
    },
    "candidate_filter": candidate_policy_manifest,
    "configuration_hash": stable_object_hash(
        {
            "columns": columns_config,
            "labeling": labeling_config,
            "validation": validation_config,
        }
    ),
}

run_manifest_path = (
    RESULT_PATHS["manifests"]
    / "04_feature_and_leakage_audit_manifest.json"
)
run_manifest_path.write_text(
    json.dumps(
        run_manifest,
        ensure_ascii=False,
        indent=2,
        default=str,
    ),
    encoding="utf-8",
)

print("Candidate policy manifest:", candidate_policy_path)
print("Run manifest:", run_manifest_path)


Candidate policy manifest: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\04_candidate_filter_policy.json
Run manifest: E:\Iran_stock_trade\financial-ml-decision-support\results\manifests\04_feature_and_leakage_audit_manifest.json


## 11. Final integrity checks

In [14]:
approved_prohibited_audit = audit_prohibited_feature_names(
    approved_model_features,
    list(feature_policy["prohibited_feature_name_patterns"]),
)

legacy_zigzag_columns = set(
    feature_policy["legacy_zigzag_columns"]
)
candidate_output_files = sorted(
    CANDIDATE_TRAIN_PATH.glob("*_train_candidates.csv")
)

assert schema_errors_df.empty, (
    "Feature schema/quality errors exist. "
    "Review results/audits/04_feature_audit_errors.csv"
)
assert candidate_errors_df.empty, (
    "Candidate generation errors exist. "
    "Review results/audits/04_candidate_generation_errors.csv"
)
assert approved_model_features, "No model features were approved."
assert not approved_prohibited_audit["prohibited_name_hit"].any()
assert legacy_zigzag_columns.isdisjoint(approved_model_features)
assert set(feature_policy["context_only_price_features"]).isdisjoint(
    approved_model_features
)
assert prefix_mismatches == 0
assert (
    zigzag_state_audit_df["high_confirmation_after_event_rows"].sum()
    == 0
)
assert (
    zigzag_state_audit_df["low_confirmation_after_event_rows"].sum()
    == 0
)
assert threshold_comparison_df["comparison_scope"].eq(
    "train_only"
).all()
assert not threshold_comparison_df[
    "automatic_threshold_selection_used"
].any()
assert np.isclose(PRIMARY_THRESHOLD, 0.15)
assert len(candidate_output_files) == len(frozen_symbols)
assert int(
    candidate_audit_df["primary_candidate_events"].sum()
) > 0

primary_row = threshold_comparison_df.loc[
    np.isclose(
        threshold_comparison_df["threshold_fraction"],
        PRIMARY_THRESHOLD,
    )
].iloc[0]


assert final_feature_errors_df.empty, (
    "Final feature-engineering errors exist. "
    "Review 04_final_feature_engineering_errors.csv"
)
assert len(MARKET_REGIME_NUMERIC_FEATURES) == 9
assert len(ENGINEERED_NUMERIC_FEATURES) == 32
assert len(CARRIED_STAGE04_NUMERIC_FEATURES) == 2
assert len(approved_model_features) == 35
assert len(FINAL_NUMERIC_FEATURES) == 34
assert len(FINAL_CATEGORICAL_FEATURES) == 1
assert set(FINAL_MODEL_FEATURES) == set(approved_model_features)
assert final_feature_quality_df["nonfinite_values"].eq(0).all()

body_quality_row = final_feature_quality_df.loc[
    final_feature_quality_df["feature"].eq("body_ratio")
].iloc[0]
assert float(body_quality_row["global_minimum"]) >= -1.0 - 1e-12
assert float(body_quality_row["global_maximum"]) <= 1.0 + 1e-12

ho_buy_quality_row = final_feature_quality_df.loc[
    final_feature_quality_df["feature"].eq("ho_buy_fraction")
].iloc[0]
ho_sell_quality_row = final_feature_quality_df.loc[
    final_feature_quality_df["feature"].eq("ho_sell_fraction")
].iloc[0]

assert float(ho_buy_quality_row["global_minimum"]) >= 0.0
assert float(ho_buy_quality_row["global_maximum"]) <= 1.0 + 1e-12
assert float(ho_sell_quality_row["global_minimum"]) >= 0.0
assert float(ho_sell_quality_row["global_maximum"]) <= 1.0 + 1e-12

assert int(final_feature_audit_df["raw_join_missing_rows"].sum()) == 0

assert inconsistent_market_rows == 0

assert TRAIN_FEATURE_HORIZON_END <= pd.Timestamp("2021-03-20")
assert canonical_market_index_df["dEven"].max() <= (
    TRAIN_FEATURE_HORIZON_END
)
assert market_regime_feature_frame["dEven"].max() <= (
    TRAIN_FEATURE_HORIZON_END
)
assert int(
    final_feature_audit_df[
        "market_feature_join_missing_rows"
    ].sum()
) == 0

market_close_quality_row = final_feature_quality_df.loc[
    final_feature_quality_df["feature"].eq(
        "market_close_location"
    )
].iloc[0]
market_range_quality_row = final_feature_quality_df.loc[
    final_feature_quality_df["feature"].eq(
        "market_range_fraction"
    )
].iloc[0]
market_drawdown_quality_row = final_feature_quality_df.loc[
    final_feature_quality_df["feature"].eq(
        "market_drawdown_60"
    )
].iloc[0]

assert float(
    market_close_quality_row["global_minimum"]
) >= -1e-12
assert float(
    market_close_quality_row["global_maximum"]
) <= 1.0 + 1e-12
assert float(
    market_range_quality_row["global_minimum"]
) >= -1e-12
assert float(
    market_drawdown_quality_row["global_maximum"]
) <= 1e-12

print("Notebook 04 integrity checks: PASSED")
print("Frozen symbols audited:", len(frozen_symbols))
print("Final pooled-model features:", len(approved_model_features))
print("Market-regime features:", len(MARKET_REGIME_NUMERIC_FEATURES))
print("Stage 04 market feature horizon end:", TRAIN_FEATURE_HORIZON_END.date())
print("Canonical market dates:", len(canonical_market_index_df))
print("Post-train market values used for feature engineering: False")
print("Market-index cross-file inconsistencies:", inconsistent_market_rows)
print("Deterministically engineered features:", len(ENGINEERED_MODEL_FEATURES))
print("Carried Stage 04 ZigZag geometry features:", len(CARRIED_STAGE04_NUMERIC_FEATURES))
print("Legacy ZigZag new_2 direct use: False")
print("Confirmation-gated ZigZag reconstruction: True")
print("Prefix-invariance mismatches:", prefix_mismatches)
print("Primary candidate threshold: 15%")
print("Threshold selection scope: train only")
print("Automatic threshold selection: False")
print("Primary candidate long events:", int(primary_row["candidate_events"]))
print(
    "Primary positive meta-label fraction:",
    round(float(primary_row["positive_meta_label_fraction"]), 6),
)
print("Unadjusted-price body_ratio using same-day priceChange: True")
print("Unseen-test used in Stage 04 decisions: False")
print(
    "Next stage: Notebook 05 — Purged anchored walk-forward design "
    "on candidate long events."
)


Notebook 04 integrity checks: PASSED
Frozen symbols audited: 499
Final pooled-model features: 35
Market-regime features: 9
Stage 04 market feature horizon end: 2021-03-17
Canonical market dates: 1690
Post-train market values used for feature engineering: False
Market-index cross-file inconsistencies: 0
Deterministically engineered features: 33
Carried Stage 04 ZigZag geometry features: 2
Legacy ZigZag new_2 direct use: False
Confirmation-gated ZigZag reconstruction: True
Prefix-invariance mismatches: 0
Primary candidate threshold: 15%
Threshold selection scope: train only
Automatic threshold selection: False
Primary candidate long events: 118464
Primary positive meta-label fraction: 0.539176
Unadjusted-price body_ratio using same-day priceChange: True
Unseen-test used in Stage 04 decisions: False
Next stage: Notebook 05 — Purged anchored walk-forward design on candidate long events.


## Review before freezing Stage 04

Inspect:

- `results/audits/04_feature_approval_audit.csv`
- `results/audits/04_legacy_zigzag_source_logic_audit.csv`
- `results/audits/04_zigzag_prefix_invariance_audit.csv`
- `results/audits/04_confirmation_gated_zigzag_state_audit.csv`
- `results/audits/04_candidate_filter_threshold_comparison.csv`
- `results/audits/04_candidate_filter_by_year.csv`
- `results/audits/04_candidate_event_audit.csv`
- `results/audits/04_feature_audit_errors.csv`
- `results/audits/04_candidate_generation_errors.csv`
- `results/manifests/04_approved_model_features.csv`
- `results/manifests/04_candidate_filter_policy.json`
- `results/manifests/04_feature_and_leakage_audit_manifest.json`

Do not proceed to Notebook 05 until the final integrity checks pass and the
candidate-event density is reviewed.

## Recommended Git checkpoint after successful review

```bash
git add -A
git commit -m "feat: freeze leakage-audited features and confirmed ZigZag candidate events"
git push origin main

git tag -a milestone-04-feature-zigzag-audit -m "Freeze model features and confirmation-gated long-event filter"
git push origin milestone-04-feature-zigzag-audit
```


## 12. Causal Market Breadth Extension

This final Stage 04 section enriches unchanged train candidate rows with original causal `started` and causal cross-sectional market breadth. No hard `started` or breadth filter is applied.


In [ ]:
from src.features.stage04_breadth_extension import (
    STAGE04_BREADTH_FEATURES,
    run_stage04_breadth_extension,
)

stage04_breadth_outputs = run_stage04_breadth_extension(
    REPOSITORY_ROOT
)

print("Stage 04 breadth extension: PASSED")
print(
    "Final approved model features:",
    stage04_breadth_outputs["summary"]["approved_features"],
)
print(
    "Added features:",
    stage04_breadth_outputs["summary"]["added_features"],
)
print(
    "Candidate identity preserved:",
    stage04_breadth_outputs["summary"][
        "candidate_identity_preserved"
    ],
)
print(
    "Hard started filter applied:",
    stage04_breadth_outputs["summary"][
        "hard_started_filter_applied"
    ],
)
print(
    "Hard breadth filter applied:",
    stage04_breadth_outputs["summary"][
        "hard_breadth_filter_applied"
    ],
)
display(
    pd.DataFrame(
        [stage04_breadth_outputs["summary"]]
    )
)
display(
    stage04_breadth_outputs["daily_breadth"].tail(10)
)
